#  Modelagem — Previsão de Churn Bancário

**Objetivo:** Treinar, comparar e avaliar modelos de classificação para prever quais clientes têm maior probabilidade de cancelar — e traduzir os resultados em impacto de negócio.

**Input:** `data/processed/churn_features.csv`  
**Output:** modelo final + relatório de métricas + insights de negócio



## 1. Importações e carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

# Modelos
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.dummy           import DummyClassifier

# Pré-processamento e validação
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline

# Métricas
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, average_precision_score
)

# XGBoost
try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except ImportError:
    XGBOOST_OK = False
    print('⚠️  XGBoost não instalado. Rode: pip install xgboost')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

COLORS = {'permaneceu': '#4C72B0', 'churn': '#DD4949'}
RANDOM_STATE = 42

print('✅ Bibliotecas carregadas!')
print(f'   XGBoost disponível: {XGBOOST_OK}')

In [ ]:
# Carregar dataset gerado pelo notebook 02
df = pd.read_csv('../data/processed/churn_features.csv')

TARGET = 'churn'
feature_cols = [c for c in df.columns if c != TARGET]

X = df[feature_cols]
y = df[TARGET]

print(f'Shape: {df.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Churn positivo: {y.sum():,} ({y.mean()*100:.1f}%)')
print(f'Churn negativo: {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)')

## 2. Divisão treino / teste



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y          # mantém proporção de churn em ambos os splits
)

print(f'Treino:  {X_train.shape[0]:,} registros — {y_train.mean()*100:.1f}% churn')
print(f'Teste:   {X_test.shape[0]:,} registros  — {y_test.mean()*100:.1f}% churn')
print()
print('⚠️  REGRA DE OURO: o conjunto de teste nunca será visto durante o treino.')
print('   Ele representa clientes novos que o modelo encontrará na vida real.')

## 3. Linha de base (Baseline)



In [ ]:
# DummyClassifier: sempre prevê a classe majoritária ("ninguém faz churn")
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

acuracia_dummy = (y_pred_dummy == y_test).mean() * 100
f1_dummy       = f1_score(y_test, y_pred_dummy, zero_division=0)

print('── BASELINE (DummyClassifier) ──────────────────────')
print(f'Acurácia: {acuracia_dummy:.1f}%')
print(f'F1-Score: {f1_dummy:.4f}')
print()
print('📝 Acurácia de 79% parece boa — mas o modelo está só dizendo')
print('   "ninguém faz churn" para todos os clientes.')
print('   Ele não identifica NENHUM cliente em risco.')
print('   Por isso usamos F1 e AUC-ROC como métricas principais.')

## 4. Regressão Logística



In [ ]:
# Pipeline: escala os dados antes de treinar (obrigatório para modelos lineares)
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
        class_weight='balanced',   # trata o desbalanceamento
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

# Cross-validation com 5 folds estratificados
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results_lr = cross_validate(
    pipe_lr, X_train, y_train, cv=cv,
    scoring=['roc_auc', 'f1', 'precision', 'recall'],
    return_train_score=True
)

print('── REGRESSÃO LOGÍSTICA — Cross-Validation (5 folds) ─')
for metric in ['roc_auc', 'f1', 'precision', 'recall']:
    val_scores = cv_results_lr[f'test_{metric}']
    print(f'{metric:12s}: {val_scores.mean():.4f} ± {val_scores.std():.4f}')

# Treinar no treino completo e avaliar no teste
pipe_lr.fit(X_train, y_train)
y_pred_lr   = pipe_lr.predict(X_test)
y_proba_lr  = pipe_lr.predict_proba(X_test)[:, 1]

print(f'\nAUC-ROC no teste: {roc_auc_score(y_test, y_proba_lr):.4f}')

## 5. Random Forest

In [ ]:
# Random Forest não precisa de escalonamento
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv_results_rf = cross_validate(
    rf, X_train, y_train, cv=cv,
    scoring=['roc_auc', 'f1', 'precision', 'recall'],
    return_train_score=True
)

print('── RANDOM FOREST — Cross-Validation (5 folds) ───────')
for metric in ['roc_auc', 'f1', 'precision', 'recall']:
    train_scores = cv_results_rf[f'train_{metric}']
    val_scores   = cv_results_rf[f'test_{metric}']
    gap = train_scores.mean() - val_scores.mean()
    print(f'{metric:12s}: treino {train_scores.mean():.4f} | val {val_scores.mean():.4f} ± {val_scores.std():.4f} | gap {gap:.4f}')

# Treinar e avaliar no teste
rf.fit(X_train, y_train)
y_pred_rf   = rf.predict(X_test)
y_proba_rf  = rf.predict_proba(X_test)[:, 1]

print(f'\nAUC-ROC no teste: {roc_auc_score(y_test, y_proba_rf):.4f}')
print()
print('📝 Observe o gap treino vs validação:')
print('   Gap pequeno (<0.05) = modelo generaliza bem')
print('   Gap grande (>0.10)  = overfitting — modelo decorou o treino')

## 6. XGBoost

In [ ]:
if XGBOOST_OK:
    # scale_pos_weight trata o desbalanceamento no XGBoost
    escala_peso = (y_train == 0).sum() / (y_train == 1).sum()

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=escala_peso,   # equivalente ao class_weight='balanced'
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0,
        n_jobs=-1
    )

    cv_results_xgb = cross_validate(
        xgb, X_train, y_train, cv=cv,
        scoring=['roc_auc', 'f1', 'precision', 'recall'],
        return_train_score=True
    )

    print('── XGBOOST — Cross-Validation (5 folds) ─────────────')
    for metric in ['roc_auc', 'f1', 'precision', 'recall']:
        train_scores = cv_results_xgb[f'train_{metric}']
        val_scores   = cv_results_xgb[f'test_{metric}']
        gap = train_scores.mean() - val_scores.mean()
        print(f'{metric:12s}: treino {train_scores.mean():.4f} | val {val_scores.mean():.4f} ± {val_scores.std():.4f} | gap {gap:.4f}')

    xgb.fit(X_train, y_train)
    y_pred_xgb  = xgb.predict(X_test)
    y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

    print(f'\nAUC-ROC no teste: {roc_auc_score(y_test, y_proba_xgb):.4f}')
else:
    print('XGBoost não disponível. Instale com: pip install xgboost')

## 7. Comparação entre modelos

In [ ]:
# ── Tabela comparativa de métricas no conjunto de teste ───────────────────
modelos = {
    'Baseline'           : (y_pred_dummy,  None),
    'Regressão Logística': (y_pred_lr,     y_proba_lr),
    'Random Forest'      : (y_pred_rf,     y_proba_rf),
}
if XGBOOST_OK:
    modelos['XGBoost'] = (y_pred_xgb, y_proba_xgb)

linhas = []
for nome, (y_pred, y_proba) in modelos.items():
    auc  = roc_auc_score(y_test, y_proba) if y_proba is not None else 0.5
    f1   = f1_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    acc  = (y_pred == y_test).mean()
    linhas.append({'Modelo': nome, 'AUC-ROC': auc, 'F1': f1,
                   'Precision': prec, 'Recall': rec, 'Acurácia': acc})

df_resultados = pd.DataFrame(linhas).set_index('Modelo')
print('── COMPARAÇÃO DE MÉTRICAS NO CONJUNTO DE TESTE ──────')
display(
    df_resultados.style
    .format('{:.4f}')
    .background_gradient(cmap='RdYlGn', subset=['AUC-ROC', 'F1', 'Recall'])
    .highlight_max(subset=['AUC-ROC', 'F1'], color='#c6efce')
)

print()
print('📝 Por que AUC-ROC e F1 são as métricas principais?')
print('   Acurácia engana: um modelo que diz "ninguém faz churn"')
print('   tem 79% de acurácia mas é inútil para o negócio.')
print('   AUC-ROC mede a capacidade de separar os dois grupos.')
print('   F1 equilibra precision (evitar alarme falso) e recall (pegar os churns).')

In [ ]:
# ── Gráfico de barras comparativo ─────────────────────────────────────────
metricas_plot = ['AUC-ROC', 'F1', 'Precision', 'Recall']
x = np.arange(len(metricas_plot))
largura = 0.18

fig, ax = plt.subplots(figsize=(12, 5))
paleta = ['#AAAAAA', '#4C72B0', '#2CA02C', '#DD4949']

for i, (nome, row) in enumerate(df_resultados.iterrows()):
    offset = (i - len(df_resultados)/2 + 0.5) * largura
    bars = ax.bar(x + offset, row[metricas_plot].values,
                  width=largura, label=nome, color=paleta[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metricas_plot)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Comparação de Modelos — Métricas no Conjunto de Teste',
             fontweight='bold', fontsize=13)
ax.legend(loc='upper right')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig('../reports/15_comparacao_modelos.png', bbox_inches='tight')
plt.show()

## 8. Análise de erros e curva ROC

In [ ]:
# ── Selecionar o melhor modelo para análise detalhada ─────────────────────

MELHOR_MODELO_NOME  = 'XGBoost' if XGBOOST_OK else 'Random Forest'
y_pred_melhor       = y_pred_xgb  if XGBOOST_OK else y_pred_rf
y_proba_melhor      = y_proba_xgb if XGBOOST_OK else y_proba_rf

print(f'Modelo selecionado para análise: {MELHOR_MODELO_NOME}')

In [ ]:
# ── Matriz de confusão ────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_melhor)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap da matriz
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Previsto: Permanece', 'Previsto: Churn'],
    yticklabels=['Real: Permanece', 'Real: Churn'],
    ax=axes[0], linewidths=1
)
axes[0].set_title(f'Matriz de Confusão — {MELHOR_MODELO_NOME}', fontweight='bold')

# Explicação visual dos 4 quadrantes
explicacoes = [
    (tn, 'Verdadeiro\nNegativo (TN)', '#4C72B0',
     'Cliente ficou\ne previmos que ficaria'),
    (fp, 'Falso\nPositivo (FP)', '#F0A500',
     'Cliente ficou\nmas previmos churn\n(custo: ação desnecessária)'),
    (fn, 'Falso\nNegativo (FN)', '#DD4949',
     'Cliente saiu\nmas não previmos\n(PIOR ERRO — cliente perdido)'),
    (tp, 'Verdadeiro\nPositivo (TP)', '#2CA02C',
     'Cliente saiu\ne previmos corretamente'),
]

axes[1].axis('off')
for i, (n, titulo, cor, desc) in enumerate(explicacoes):
    x_pos = 0.05 + (i % 2) * 0.5
    y_pos = 0.55 - (i // 2) * 0.5
    axes[1].add_patch(plt.Rectangle((x_pos, y_pos), 0.42, 0.38,
                                     facecolor=cor, alpha=0.15,
                                     transform=axes[1].transAxes))
    axes[1].text(x_pos + 0.21, y_pos + 0.27, f'{n:,}',
                 ha='center', va='center', fontsize=18, fontweight='bold',
                 color=cor, transform=axes[1].transAxes)
    axes[1].text(x_pos + 0.21, y_pos + 0.17, titulo,
                 ha='center', va='center', fontsize=9, fontweight='bold',
                 transform=axes[1].transAxes)
    axes[1].text(x_pos + 0.21, y_pos + 0.06, desc,
                 ha='center', va='center', fontsize=8, color='gray',
                 transform=axes[1].transAxes)

axes[1].set_title('O que cada número significa?', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/16_matriz_confusao.png', bbox_inches='tight')
plt.show()

print(f'\nVerdadeiro Negativo  (TN): {tn:>5,} — acertou: cliente ficou')
print(f'Falso Positivo       (FP): {fp:>5,} — errou:   disse que sairia, mas ficou')
print(f'Falso Negativo       (FN): {fn:>5,} — errou:   disse que ficaria, mas saiu ← PIOR')
print(f'Verdadeiro Positivo  (TP): {tp:>5,} — acertou: cliente saiu')

In [ ]:
# ── Curva ROC — todos os modelos juntos ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
modelos_proba = {
    'Regressão Logística': y_proba_lr,
    'Random Forest'      : y_proba_rf,
}
if XGBOOST_OK:
    modelos_proba['XGBoost'] = y_proba_xgb

cores_roc = ['#4C72B0', '#2CA02C', '#DD4949']

for (nome, proba), cor in zip(modelos_proba.items(), cores_roc):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=cor, linewidth=2, label=f'{nome} (AUC={auc:.3f})')

axes[0].plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Baseline (AUC=0.500)')
axes[0].fill_between([0,1],[0,1], alpha=0.05, color='gray')
axes[0].set_xlabel('Taxa de Falso Positivo (FPR)')
axes[0].set_ylabel('Taxa de Verdadeiro Positivo (TPR / Recall)')
axes[0].set_title('Curva ROC', fontweight='bold')
axes[0].legend(fontsize=9)

# Curva Precision-Recall (mais informativa com dados desbalanceados)
for (nome, proba), cor in zip(modelos_proba.items(), cores_roc):
    prec_c, rec_c, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec_c, prec_c, color=cor, linewidth=2, label=f'{nome} (AP={ap:.3f})')

baseline_pr = y_test.mean()
axes[1].axhline(baseline_pr, color='gray', linestyle='--', linewidth=1,
                label=f'Baseline ({baseline_pr:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Avaliação de Performance — Curvas de Discriminação',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/17_curvas_roc_pr.png', bbox_inches='tight')
plt.show()

print('📝 AUC-ROC próximo de 1.0 = modelo separa bem os dois grupos.')
print('   Precision-Recall é mais honesta com dados desbalanceados.')

In [ ]:
# ── Distribuição das probabilidades previstas ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

for label, cor, nome_label in [(0, COLORS['permaneceu'], 'Permaneceu'), 
                                (1, COLORS['churn'], 'Churn')]:
    mask = y_test == label
    ax.hist(y_proba_melhor[mask], bins=40, alpha=0.6, color=cor,
            label=nome_label, density=True)

ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold 0.5')
ax.set_xlabel('Probabilidade prevista de churn')
ax.set_ylabel('Densidade')
ax.set_title(f'Distribuição das Probabilidades — {MELHOR_MODELO_NOME}', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/18_distribuicao_probabilidades.png', bbox_inches='tight')
plt.show()

print('📝 Um bom modelo separa claramente as duas distribuições.')
print('   Se houver sobreposição grande no centro, o modelo está inseguro.')

## 9. Impacto de negócio — quanto custa cada erro?



In [ ]:
# ── Premissas de negócio (estimativas realistas para um banco europeu) ────
# Ajuste conforme o contexto real do projeto

RECEITA_ANUAL_POR_CLIENTE   = 1200   # €/ano — receita média por cliente ativo
CUSTO_ACAO_RETENCAO         = 150    # €    — cupom, ligação, oferta personalizada
TAXA_SUCESSO_RETENCAO       = 0.30   # 30%  — % dos clientes em risco que ficam se abordados

print('── PREMISSAS DE NEGÓCIO ─────────────────────────────')
print(f'Receita anual por cliente:   € {RECEITA_ANUAL_POR_CLIENTE:,.0f}')
print(f'Custo de ação de retenção:   € {CUSTO_ACAO_RETENCAO:,.0f}')
print(f'Taxa de sucesso da retenção: {TAXA_SUCESSO_RETENCAO*100:.0f}%')
print()
print('📝 Essas premissas viriam do time de negócio em um cenário real.')
print('   Aqui usamos estimativas conservadoras para demonstrar o raciocínio.')

In [ ]:
# ── Cálculo do impacto financeiro ─────────────────────────────────────────
cm_melhor = confusion_matrix(y_test, y_pred_melhor)
tn_m, fp_m, fn_m, tp_m = cm_melhor.ravel()

# Custo do Falso Negativo: cliente que ia sair mas não previmos
# → perdemos a receita desse cliente para sempre
custo_fn = fn_m * RECEITA_ANUAL_POR_CLIENTE

# Custo do Falso Positivo: cliente que ficaria mas abordamos com retenção
# → gastamos a ação de retenção sem necessidade
custo_fp = fp_m * CUSTO_ACAO_RETENCAO

# Valor do Verdadeiro Positivo: clientes em risco que identificamos
# → de cada TP, TAXA_SUCESSO ficam por causa da ação
clientes_salvos = tp_m * TAXA_SUCESSO_RETENCAO
receita_salva   = clientes_salvos * RECEITA_ANUAL_POR_CLIENTE
custo_acoes_tp  = tp_m * CUSTO_ACAO_RETENCAO
lucro_retencao  = receita_salva - custo_acoes_tp

impacto_liquido = lucro_retencao - custo_fn - custo_fp

print('='*58)
print('        ANÁLISE DE IMPACTO FINANCEIRO — CONJUNTO DE TESTE')
print('='*58)
print(f'\n🔴 Falsos Negativos (clientes perdidos sem aviso):')
print(f'   {fn_m} clientes × € {RECEITA_ANUAL_POR_CLIENTE:,} = -€ {custo_fn:,.0f}')
print(f'\n🟡 Falsos Positivos (ações de retenção desnecessárias):')
print(f'   {fp_m} clientes × € {CUSTO_ACAO_RETENCAO:,} = -€ {custo_fp:,.0f}')
print(f'\n🟢 Verdadeiros Positivos (clientes em risco identificados):')
print(f'   {tp_m} identificados × {TAXA_SUCESSO_RETENCAO*100:.0f}% sucesso = {clientes_salvos:.0f} clientes salvos')
print(f'   Receita salva:  +€ {receita_salva:,.0f}')
print(f'   Custo das ações: -€ {custo_acoes_tp:,.0f}')
print(f'   Lucro líquido:   +€ {lucro_retencao:,.0f}')
print(f'\n{"─"*58}')
print(f'IMPACTO LÍQUIDO DO MODELO: € {impacto_liquido:+,.0f}')
print('='*58)

# Escalar para a base completa de clientes
fator_escala = len(y) / len(y_test)
print(f'\n📊 Extrapolando para os {len(y):,} clientes totais:')
print(f'   Impacto estimado: € {impacto_liquido * fator_escala:+,.0f}/ano')

In [ ]:
# ── Gráfico de impacto financeiro ─────────────────────────────────────────
categorias = ['Receita salva\n(TP)', 'Custo ações\n(TP)', 
              'Clientes perdidos\n(FN)', 'Ações desnec.\n(FP)', 'Impacto líquido']
valores    = [receita_salva, -custo_acoes_tp, -custo_fn, -custo_fp, impacto_liquido]
cores_bar  = ['#2CA02C', '#F0A500', '#DD4949', '#DD8888',
              '#2CA02C' if impacto_liquido > 0 else '#DD4949']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(categorias, valores, color=cores_bar, width=0.5, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)

for bar, val in zip(bars, valores):
    ypos = bar.get_height() + (500 if val >= 0 else -3000)
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'€ {val:+,.0f}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('Impacto Financeiro do Modelo de Churn — Conjunto de Teste',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Impacto (€)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'€ {x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/19_impacto_financeiro.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Análise de threshold: qual corte de probabilidade maximiza o lucro? ───


thresholds  = np.arange(0.1, 0.9, 0.02)
lucros      = []
f1s         = []

for t in thresholds:
    y_pred_t = (y_proba_melhor >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()

    recv  = tp_t * TAXA_SUCESSO_RETENCAO * RECEITA_ANUAL_POR_CLIENTE
    cst   = (tp_t + fp_t) * CUSTO_ACAO_RETENCAO
    perd  = fn_t * RECEITA_ANUAL_POR_CLIENTE
    lucros.append(recv - cst - perd)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

idx_melhor_lucro = np.argmax(lucros)
idx_melhor_f1    = np.argmax(f1s)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.plot(thresholds, lucros, color='#2CA02C', linewidth=2, label='Impacto financeiro (€)')
ax2.plot(thresholds, f1s,    color='#4C72B0', linewidth=2, linestyle='--', label='F1-Score')

ax1.axvline(thresholds[idx_melhor_lucro], color='#2CA02C', linestyle=':', linewidth=1.5,
            label=f'Melhor threshold financeiro: {thresholds[idx_melhor_lucro]:.2f}')
ax1.axvline(thresholds[idx_melhor_f1], color='#4C72B0', linestyle=':', linewidth=1.5,
            label=f'Melhor threshold F1: {thresholds[idx_melhor_f1]:.2f}')
ax1.axvline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Threshold padrão: 0.50')

ax1.set_xlabel('Threshold de probabilidade')
ax1.set_ylabel('Impacto financeiro (€)', color='#2CA02C')
ax2.set_ylabel('F1-Score', color='#4C72B0')
ax1.set_title('Threshold Ótimo: F1 vs Impacto Financeiro', fontweight='bold', fontsize=13)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='lower left')

plt.tight_layout()
plt.savefig('../reports/20_threshold_otimo.png', bbox_inches='tight')
plt.show()

print(f'Threshold que maximiza lucro: {thresholds[idx_melhor_lucro]:.2f}')
print(f'Threshold que maximiza F1:    {thresholds[idx_melhor_f1]:.2f}')
print(f'Threshold padrão:             0.50')
print()
print('📝 Threshold menor = modelo mais agressivo na detecção de churn.')
print('   A escolha final depende do custo de retenção vs receita do cliente.')

## 10. Conclusões 

In [ ]:
# ── Relatório final ────────────────────────────────────────────────────────
melhor_auc = roc_auc_score(y_test, y_proba_melhor)
melhor_f1  = f1_score(y_test, y_pred_melhor)
threshold_otimo = thresholds[idx_melhor_lucro]

print('='*60)
print('              RELATÓRIO FINAL — MODELAGEM DE CHURN')
print('='*60)
print(f'\n🏆 Melhor modelo: {MELHOR_MODELO_NOME}')
print(f'   AUC-ROC:  {melhor_auc:.4f}')
print(f'   F1-Score: {melhor_f1:.4f}')
print(f'\n💰 Impacto financeiro estimado:')
print(f'   Teste ({len(y_test):,} clientes): € {impacto_liquido:+,.0f}')
print(f'   Base total ({len(y):,} clientes): € {impacto_liquido * fator_escala:+,.0f}/ano')
print(f'\n⚙️  Threshold recomendado: {threshold_otimo:.2f} (máximo impacto financeiro)')
print()
print('📌 PRINCIPAIS ACHADOS:')
print('   1. Idade é o preditor mais forte — clientes acima de 45 anos')
print('      têm risco de churn significativamente maior')
print('   2. Membros inativos têm taxa de churn ~2x maior que ativos')
print('   3. Clientes alemães têm risco ~2x maior que franceses')
print('   4. Clientes com 3-4 produtos têm comportamento anômalo de churn')
print()


In [ ]:
# ── Salvar modelo final ───────────────────────────────────────────────────
import pickle, os

os.makedirs('../models', exist_ok=True)

modelo_final = xgb if XGBOOST_OK else rf
with open('../models/modelo_churn_final.pkl', 'wb') as f:
    pickle.dump(modelo_final, f)

# Salvar também as features usadas
with open('../models/features_modelo.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print('✅ Modelo salvo em models/modelo_churn_final.pkl')
print('✅ Features salvas em models/features_modelo.txt')
print()
print('Para carregar o modelo depois:')
print("  with open('../models/modelo_churn_final.pkl', 'rb') as f:")
print('      modelo = pickle.load(f)')
print('  probabilidade = modelo.predict_proba(X_novo)[:, 1]')